### Faiss 

Facebook AI similarity search is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size , up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 200, chunk_overlap=30)

docs = text_splitter.split_documents(documents)

Created a chunk of size 208, which is longer than the specified 200
Created a chunk of size 210, which is longer than the specified 200
Created a chunk of size 269, which is longer than the specified 200
Created a chunk of size 253, which is longer than the specified 200
Created a chunk of size 271, which is longer than the specified 200
Created a chunk of size 290, which is longer than the specified 200
Created a chunk of size 271, which is longer than the specified 200
Created a chunk of size 240, which is longer than the specified 200


In [4]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Machine Learning is a subset of AI that enables computers to learn from data without being explicitly programmed. By analyzing large datasets, machine learning models can identify trends and make predictions.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Natural Language Processing (NLP) focuses on enabling computers to understand and generate human language. Applications of NLP include chatbots, language translation, sentiment analysis, and text summarization.'),
 Document(metadata={'source': 'speech.txt'}, page_content="Retrieval-Augmented Generation (RAG) combines information retrieval techniques with large language models. Instead of relying solely on the mod

In [5]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs, embeddings)
db

In [6]:
# Query
query = "What is Artificial Intelligence?"
docs = db.similarity_search(query)
docs[0].page_content

'Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'

As a Retriever

we can also convert the vectorstore into a Retriever class. This allows us to easily use in other LangChain methods, 

which largely work with retrievers

In [7]:
retriever = db.as_retriever()
docs2 = retriever.invoke(query)
docs2[0].page_content

'Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'

Similarity search with score 

There are some Faiss specific methods. One of them is similarity_search_with_score which allows us to return not only the documents but also the distance score of the query to them. Lowered score is better

In [12]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='e615333e-73d4-4b64-9706-64ead7694b98', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'),
  np.float32(0.40107006)),
 (Document(id='827bbb54-46fb-48b8-ac14-2cf749b6e3ac', metadata={'source': 'speech.txt'}, page_content='Machine Learning is a subset of AI that enables computers to learn from data without being explicitly programmed. By analyzing large datasets, machine learning models can identify trends and make predictions.'),
  np.float32(0.6181884)),
 (Document(id='f4fc97d3-20d3-4d9a-851e-6faa20a7c598', metadata={'source': 'speech.txt'}, page_content='Agents are AI systems capable of using tools to accomplish tasks. An agent can analyze a user request, determine which tool to use, execute the tool, and use the results to generate a final response. Examples of tools include web s

In [15]:
embedding_vector = embeddings.embed_query(query)
doc3 = db.similarity_search_by_vector(embedding_vector)
doc3[0].page_content

'Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'

In [16]:
# SAVING AND LOADING
db.save_local("faiss_index")

In [19]:
df = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs4 = df.similarity_search(query)
docs4

[Document(id='e615333e-73d4-4b64-9706-64ead7694b98', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence (AI) is transforming the way people interact with technology. AI systems can analyze data, recognize patterns, and assist in decision-making across various industries.'),
 Document(id='827bbb54-46fb-48b8-ac14-2cf749b6e3ac', metadata={'source': 'speech.txt'}, page_content='Machine Learning is a subset of AI that enables computers to learn from data without being explicitly programmed. By analyzing large datasets, machine learning models can identify trends and make predictions.'),
 Document(id='f4fc97d3-20d3-4d9a-851e-6faa20a7c598', metadata={'source': 'speech.txt'}, page_content='Agents are AI systems capable of using tools to accomplish tasks. An agent can analyze a user request, determine which tool to use, execute the tool, and use the results to generate a final response. Examples of tools include web search, calculators, databases, and APIs.'),
 Document(id